# OmniSpeak (Colab)

Đọc văn bản, nhân bản giọng nói, lưu thư viện giọng — chạy trên Google Colab, dùng model [OmniVoice](https://github.com/k2-fsa/OmniVoice) (`k2-fsa/OmniVoice`, Apache-2.0).

Code app (frontend + backend + script) nằm trên GitHub, notebook chỉ `git clone`/`git pull` về rồi gọi — không nhúng code app trong notebook.

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy các cell theo thứ tự từ trên xuống — mọi cell đều an toàn khi chạy lại.

## Cài đặt

In [ ]:
# 1. GPU check
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Không có GPU — Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu.")

In [ ]:
# 2. Lấy code (frontend + backend + scripts) từ GitHub — luôn đồng bộ bản mới nhất
import importlib
import os
import subprocess
import sys

GITHUB_USER = "trkhanh8312-make"  # @param {type:"string"}
GITHUB_REPO = "omnispeak"  # @param {type:"string"}
GITHUB_BRANCH = "main"  # @param {type:"string"}

REPO_DIR = "/content/omnispeak_repo"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

def _sh(cmd, what=""):
    print("\n$", cmd)
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise SystemExit(f"Lỗi: {what or cmd}")

# Nếu thư mục cũ đang trỏ tới repo khác (đổi GITHUB_USER/GITHUB_REPO) thì xoá và clone lại
existing_remote = None
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    r = subprocess.run(f"git -C {REPO_DIR} remote get-url origin", shell=True,
                        capture_output=True, text=True)
    existing_remote = r.stdout.strip()

if existing_remote and existing_remote != REPO_URL:
    print(f"Repo cũ ({existing_remote}) khác REPO_URL hiện tại — xoá và clone lại.")
    _sh(f"rm -rf {REPO_DIR}", "xoá repo cũ")
    existing_remote = None

if existing_remote:
    _sh(f"git -C {REPO_DIR} fetch origin {GITHUB_BRANCH}", "git fetch")
    _sh(f"git -C {REPO_DIR} reset --hard origin/{GITHUB_BRANCH}", "git reset --hard")
else:
    _sh(f"git clone --branch {GITHUB_BRANCH} {REPO_URL} {REPO_DIR}", "git clone")

FRONTEND_DIR = os.path.join(REPO_DIR, "frontend")
APP_DIR = os.path.join(REPO_DIR, "backend")
SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")

for required in (
    os.path.join(FRONTEND_DIR, "index.html"),
    os.path.join(APP_DIR, "backend.py"),
    os.path.join(APP_DIR, "requirements.txt"),
    os.path.join(SCRIPTS_DIR, "colab_utils.py"),
    os.path.join(SCRIPTS_DIR, "install_deps.py"),
    os.path.join(SCRIPTS_DIR, "download_model.py"),
    os.path.join(SCRIPTS_DIR, "start_backend.py"),
):
    if not os.path.isfile(required):
        raise SystemExit(
            f"Không thấy {required}.\n"
            "Kiểm tra lại: file đã push lên GitHub chưa, và đường dẫn trong repo có đúng không."
        )

if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

def load(name):
    """Import module trong scripts/, luôn nạp lại bản vừa đồng bộ."""
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod)

print(f"Đã đồng bộ code từ {REPO_URL} (nhánh {GITHUB_BRANCH}).")

In [ ]:
# 3. Cài đặt (code ở scripts/install_deps.py trên GitHub)
load("colab_utils")
load("install_deps").main(requirements_path=os.path.join(APP_DIR, "requirements.txt"))

In [ ]:
# 4. Mount Google Drive (tuỳ chọn) — lưu bền vững giọng nói + model đã tải
import os

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/omnispeak_data"
HF_DIR = "/content/drive/MyDrive/omnispeak_hf_cache"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(HF_DIR, exist_ok=True)
os.environ["OMNISPEAK_DATA_DIR"] = DATA_DIR
os.environ["HF_HOME"] = HF_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"  # Drive (FUSE) không hỗ trợ symlink
print("Giọng nói + model sẽ lưu bền vững trên Drive.")

In [ ]:
# 5. Tải trước model (bỏ qua cũng được — sẽ tự tải khi khởi động backend)
load("download_model").main()

In [ ]:
# 6. Khởi động backend (code ở scripts/start_backend.py trên GitHub)
info = load("start_backend").start(
    app_dir=APP_DIR,
    frontend_dir=FRONTEND_DIR,
    force_restart=True,
)

In [ ]:
# 7. Mở giao diện web
from google.colab import output

output.serve_kernel_port_as_window(3900)

### Tổng kết

Danh sách giọng nói đã lưu.

In [ ]:
# Tổng kết
import requests

try:
    profiles = requests.get("http://127.0.0.1:3900/profiles", timeout=15).json()
    print(f"Giọng đã lưu ({len(profiles)}):")
    for p in profiles:
        print(" ", p["id"], p.get("name"))
except Exception as e:
    print("Không lấy được danh sách:", e)

## Xử lý sự cố

- **`device: cpu` hoặc generate chậm** — bật GPU: Runtime → Change runtime type → T4 GPU.
- **Sửa code (`frontend/index.html`, `backend/backend.py`, hoặc bất kỳ file nào trong `scripts/`) xong mà chạy vẫn y nguyên** — push code mới lên GitHub, rồi chạy lại cell 2 (đồng bộ code) và cell 6 (`force_restart=True` tự nạp lại, không cần tự kill process). Nếu chỉ sửa `scripts/install_deps.py`, `backend/requirements.txt` hoặc `scripts/download_model.py` thì chạy lại cell 2 rồi cell 3 hoặc cell 5 tương ứng.
- **Muốn cài lại package từ đầu (bỏ qua cơ chế skip-nếu-đã-cài)** — chạy `load("install_deps").main(requirements_path=os.path.join(APP_DIR, "requirements.txt"), force=True)`.
- **Đổi sang repo/nhánh khác** — sửa `GITHUB_USER`/`GITHUB_REPO`/`GITHUB_BRANCH` ngay trên form của cell 2 rồi chạy lại; cell tự xoá thư mục cũ nếu remote khác.
- **Muốn giữ giọng nói + model qua các phiên sau** — chạy cell 4 (mount Drive) trước cell 5.
- **Tab UI trắng hoặc lỗi** — chạy lại cell 6 rồi cell 7. Cho phép pop-up cho `colab.research.google.com`; hoặc đổi cell 7 sang `output.serve_kernel_port_as_iframe(3900)` để nhúng UI ngay trong notebook.
- **Không đồng bộ được code ở cell 2** — kiểm tra repo có để public không (git clone qua HTTPS không đọc được repo private nếu chưa cấu hình token), và tên nhánh `GITHUB_BRANCH` có đúng không.
- **Xem log backend** — `/content/omnispeak_backend.log`.